In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def run_numerical_stability_experiment():
    print("="*60)
    print(" 실험: KL-Divergence 입력에 따른 수치적 안정성 테스트")
    print("="*60)

    # 1. 극단적인 상황 연출 (매우 작거나 큰 로짓 값)
    # 1000.0과 같이 큰 값은 지수함수(exp)를 통과할 때 Overflow를,
    # -1000.0과 같이 작은 값은 Underflow를 유발합니다.
    student_logits = torch.tensor([[1000.0, -1000.0, 0.0]], dtype=torch.float32)
    teacher_logits = torch.tensor([[10.0, -5.0, 2.0]], dtype=torch.float32)

    T = 2.0 # Temperature

    # Teacher 분포는 안전하게 일반 Softmax 적용 (P 분포)
    p_teacher = F.softmax(teacher_logits / T, dim=1)

    print(f"[입력 데이터]\nStudent Logits: {student_logits}")
    print(f"Teacher Softmax (P): {p_teacher}\n")

    # -------------------------------------------------------------
    # 방법 A: 직관적이지만 위험한 방식 (Softmax 후 외부에서 log 취하기)
    # -------------------------------------------------------------
    print("--- 방법 A: F.softmax() 적용 후 torch.log() 취하기 ---")
    try:
        # Student의 확률 분포 Q를 구함
        q_student_naive = F.softmax(student_logits / T, dim=1)
        # 로그를 취함
        log_q_naive = torch.log(q_student_naive)

        print(f"Naive Q (Softmax): {q_student_naive}")
        print(f"Naive log(Q): {log_q_naive}")

        # PyTorch KLDivLoss는 첫 번째 인자에 이미 log가 취해진 값을 원하므로 log_target=False(기본값)로 계산
        loss_fn_naive = nn.KLDivLoss(reduction='batchmean')
        loss_a = loss_fn_naive(log_q_naive, p_teacher)
        print(f"▶ 방법 A 결과 Loss: {loss_a.item()}")

    except Exception as e:
        print(f"❌ 방법 A 오류 발생: {e}")

    print("\n" + "-"*50 + "\n")

    # -------------------------------------------------------------
    # 방법 B: 파이토치 권장 방식 (Log-Sum-Exp 트릭이 내재된 log_softmax)
    # -------------------------------------------------------------
    print("--- 방법 B: F.log_softmax() 직접 사용하기 ---")

    log_q_stable = F.log_softmax(student_logits / T, dim=1)
    print(f"Stable log(Q): {log_q_stable}")

    loss_fn_stable = nn.KLDivLoss(reduction='batchmean')
    loss_b = loss_fn_stable(log_q_stable, p_teacher)
    print(f"▶ 방법 B 결과 Loss: {loss_b.item()}")

    print("\n" + "="*60)
    print(" 결론: 방법 A는 -inf 가 발생하여 정상적인 학습이 불가능할 수 있지만,")
    print("       방법 B(log_softmax)는 동일한 값에서도 안정적인 음수 로그값을 반환함.")
    print("="*60)

if __name__ == "__main__":
    run_numerical_stability_experiment()

 실험: KL-Divergence 입력에 따른 수치적 안정성 테스트
[입력 데이터]
Student Logits: tensor([[ 1000., -1000.,     0.]])
Teacher Softmax (P): tensor([[9.8148e-01, 5.4284e-04, 1.7976e-02]])

--- 방법 A: F.softmax() 적용 후 torch.log() 취하기 ---
Naive Q (Softmax): tensor([[1., 0., 0.]])
Naive log(Q): tensor([[0., -inf, -inf]])
▶ 방법 A 결과 Loss: inf

--------------------------------------------------

--- 방법 B: F.log_softmax() 직접 사용하기 ---
Stable log(Q): tensor([[    0., -1000.,  -500.]])
▶ 방법 B 결과 Loss: 9.436394691467285

 결론: 방법 A는 -inf 가 발생하여 정상적인 학습이 불가능할 수 있지만,
       방법 B(log_softmax)는 동일한 값에서도 안정적인 음수 로그값을 반환함.
